## 5.0 Introduction

In Lecture 4 we built k-means from scratch: we defined the objective function $J^{\text{clust}}$, derived Lloyd's algorithm, and ran it by hand on a five-point toy dataset. We know exactly what the algorithm is doing at every step.

Now we scale up. Real datasets have hundreds of rows and many features — running Lloyd's algorithm by hand is out of the question. In this lecture we use **Scikit-learn**, Python's standard machine learning library, to run k-means in a single function call. Along the way we'll introduce the vocabulary that machine learning practitioners use to describe what's happening: *model*, *training*, *objective function*, *hyperparameter*, *ensemble*. These terms will appear in every ML course and textbook you encounter, and k-means is a clean setting to define them precisely.

We also face a new problem: the Iris data lives in $\mathbb{R}^4$, and we can't plot 4-dimensional data directly. We'll use **Principal Component Analysis (PCA)** to project the data down to $\mathbb{R}^2$ so we can see the cluster structure we found.

By the end of this lecture you will be able to:

- Use Scikit-learn to run k-means on a standardized real dataset
- Define model, training, objective function, hyperparameter, and ensemble in the k-means context
- Explain why multiple random restarts (`n_init`) are necessary
- Use PCA to reduce dimensionality and visualize clustering results
- Evaluate k-means output against known ground-truth labels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 5.1 Machine Learning Vocabulary

k-means is our first complete machine learning algorithm, so this is a good moment to define some terms precisely. These definitions apply far beyond k-means.

**Model.** A model is a mathematical object that takes input data and produces output — in this case, a cluster assignment for each data vector. The k-means model is defined by its $k$ centroid vectors $\mathbf{c}_1, \ldots, \mathbf{c}_k$. Once we know the centroids, we can assign any data vector to a cluster.

**Training.** Training is the process of finding the model parameters that minimize an objective function on a dataset. For k-means, training means running Lloyd's algorithm to find the centroid vectors that minimize $J^{\text{clust}}$. The word "training" is borrowed from the idea of a model *learning* from data — the data teaches the model where the clusters are.

**Objective function.** The objective function is the quantity the algorithm is trying to minimize (or maximize). For k-means it is $J^{\text{clust}}$ — the mean squared distance from each point to its assigned centroid. Every iteration of Lloyd's algorithm is guaranteed to reduce $J^{\text{clust}}$ or leave it unchanged. This is what makes the algorithm well-defined: it has a clear goal, and it makes measurable progress toward it.

**Hyperparameter.** A hyperparameter is a setting you choose *before* training begins — it is not learned from the data. For k-means, $k$ (the number of clusters) is the key hyperparameter. Choosing $k$ requires judgment: too small and you miss real structure; too large and you overfit noise. The elbow plot is one tool for making this choice.

**Ensemble.** An ensemble combines multiple models to produce a better result than any single model would. In k-means, Lloyd's algorithm can converge to different local minima depending on the initial centroids. The standard fix is to run the algorithm many times with different random initializations and keep the solution with the lowest $J^{\text{clust}}$. This is a simple ensemble: many models, one winner. Scikit-learn's `n_init` parameter controls how many runs are used.

**Question.** $k$ is a hyperparameter — you set it before training. The centroid vectors $\mathbf{c}_1, \ldots, \mathbf{c}_k$ are model parameters — they are learned during training. What is the difference between a hyperparameter and a model parameter?

## 5.2 k-Means on Iris with Scikit-learn

Running Lloyd's algorithm by hand was instructive, but impractical for real datasets. Scikit-learn's `KMeans` runs the full algorithm — including multiple independent runs with different initializations — in a single function call.

There is an important subtlety worth flagging: highly optimized libraries like Scikit-learn often run a more sophisticated version of an algorithm than the textbook one you learn from scratch. Lloyd's algorithm, which we implemented in Lecture 4, computes distances from every point to every centroid at every iteration — simple and correct, but slow by modern big-data standards. Scikit-learn defaults to **Elkan's algorithm**, which uses the [triangle inequality](https://en.wikipedia.org/wiki/Triangle_inequality) to skip redundant distance calculations and can be dramatically faster on large datasets. For initialization, it uses **k-means++** rather than picking centroids at random — a smarter seeding strategy that tends to find better solutions with fewer independent runs. The mathematical result is the same: the same objective function $J^{\text{clust}}$, the same convergence guarantee, the same final clusters. But the path to get there is different. This pattern — learn the simple version to understand the idea, then use the optimized library version in practice — is common across all of machine learning.

First, let's reload the Iris data and standardize it exactly as we did in Lecture 3.

In [ ]:
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/ajorgen1/Linear-Algebra-With-Applications/main/datasets/iris.csv'
df = pd.read_csv(url)

X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
y = pd.Categorical(df['species']).codes   # 0=setosa, 1=versicolor, 2=virginica
species_names = df['species'].unique()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print('X_std shape:', X_std.shape)
print('Column norms (should all be sqrt(150)):',
      np.round([np.linalg.norm(X_std[:, j]) for j in range(4)], 4))

### Ensemble Models and `n_init`

Because Lloyd's algorithm starts with randomly initialized centroids, different initializations can lead to different final groupings. This is a real problem — the algorithm is only guaranteed to find a *local* minimum of $J^{\text{clust}}$, not the global one.

The standard solution is an **ensemble model**: run the algorithm $m$ independent times with different random initializations, then keep the solution with the lowest $J^{\text{clust}}$. The rest are discarded. The `n_init` parameter controls how many independent runs are used.

This is not a voting ensemble — the models do not pool their assignments or take a majority vote. Each run produces a complete, independent solution, and the single best one wins. Contrast this with random forests, where many decision trees each vote on a prediction and the majority rules. k-means ensemble is winner-takes-all; random forest is majority-rules.

We also set `random_state` to make results reproducible — it seeds the random number generator so the same initialization sequence is used every time.

**Question.** Why do we keep the run with the lowest $J^{\text{clust}}$ rather than, say, the run that finishes in the fewest iterations?

In [ ]:
# Run k-means: 3 clusters, 10 random restarts, reproducible seed
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(X_std)

print('Cluster labels (first 150):', labels[:150])
print('Cluster label counts:', np.bincount(labels))

In [ ]:
# The learned centroids (in standardized space)
centers = kmeans.cluster_centers_
print('Cluster centroids (standardized):')
print(np.round(centers, 4))

**Question.** Setosa is typically recovered perfectly. Versicolor and Virginica are harder to separate — they overlap in feature space. Does this match what you observed in the Lecture 3 scatter plots?

In [ ]:
# All 6 pairwise scatter plots of the 4 Iris features, colored by k-means cluster
import itertools

colors = ['steelblue', 'tomato', 'seagreen']
pairs = list(itertools.combinations(range(4), 2))   # 6 unique pairs

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for ax, (j1, j2) in zip(axes.flat, pairs):
    for c in range(3):
        mask = (labels == c)
        ax.scatter(X_std[mask, j1], X_std[mask, j2],
                   color=colors[c], alpha=0.7,
                   edgecolors='white', linewidth=0.4, s=40,
                   label=f'Cluster {c}')
    # Overlay centroids as white stars with black edges
    for c in range(3):
        ax.scatter(centers[c, j1], centers[c, j2],
                   marker='*', s=350, color=colors[c],
                   edgecolors='black', linewidth=0.8, zorder=5)
    ax.set_xlabel(feature_names[j1], fontsize=9)
    ax.set_ylabel(feature_names[j2], fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

axes[0, 0].legend(fontsize=8)
plt.suptitle('Iris: All 6 Feature-Pair Scatter Plots — Colored by k-Means Cluster',
             fontsize=12)
plt.tight_layout()
plt.show()

With 4 features, the number of unique pairs is given by the combination formula:

$$\binom{4}{2} = \frac{4!}{2!(4-2)!} = \frac{4!}{2! \cdot 2!} = \frac{24}{2 \cdot 2} = \frac{24}{4} = 6$$

exactly the 6 plots above. In general, comparing $n$ features requires $\binom{n}{2}$ scatter plots.

**Question.** How many scatter plots would you need to compare all pairs of 5 features?

**Question.** Looking at the scatter plots above, the three clusters appear to overlap in several of the feature-pair views. Does this mean k-means is failing at what it's supposed to do — finding well-separated groups? Or is something else going on?

## 5.3 The Visualization Problem

We have cluster labels. Now we want to plot the results. But $\mathbf{x}_i \in \mathbb{R}^4$ — we cannot directly scatter-plot 4-dimensional data. In Lecture 3 we worked around this by picking two features to plot (petal length vs. petal width). That's fine when you know which two features to look at, but it scales poorly: with 4 features there are $\binom{4}{2} = 6$ unique pairs to examine, and with 10 features there are 45.

**Question.** How many scatter plots would you need to compare all pairs of 4 feature vectors?

What we really want is a method that finds **the two directions that capture the most variation in the data** — directions that together give the best 2D summary of the full 4D dataset.

## 5.4 What Is PCA?

We have 150 Iris flowers, each described by four measurements. That means each flower is a point in 4-dimensional space — impossible to draw. But most of the interesting structure in the data (the three species clusters) probably doesn't require all four dimensions to see. PCA finds a way to compress the data down to 2 dimensions while losing as little information as possible.

The key idea is **spread**. Imagine shining a flashlight on a cloud of points from different angles and watching the shadow it casts on a wall. Some angles produce a big, spread-out shadow that shows the shape of the cloud well. Other angles collapse everything into a tight blob and lose the structure entirely. PCA finds the angle — the direction — that produces the most spread-out shadow. That is the **first principal component**. It then finds the best perpendicular direction for a second axis, and so on.

The result is a new 2D coordinate system — one axis pointing in the direction of maximum spread, the other in the direction of maximum remaining spread — and every flower gets a new $(x, y)$ coordinate in that system. That 2D picture is the best possible flat summary of the 4D data.

In Homework 2 you projected vectors onto a direction and recovered the component in that direction. PCA is the same idea, scaled up: instead of projecting one vector onto one direction, we project an entire dataset onto the directions that preserve the most information.

**PCA as a big idea in data visualization.** If your major is Data Science or a visualization-heavy field, you will encounter this idea constantly under different names. t-SNE and UMAP are nonlinear cousins of PCA used to visualize high-dimensional datasets like gene expression profiles or word embeddings. Cartographers face the same problem every time they flatten a globe onto a map — the 3D surface of the Earth must be projected to 2D, and every projection distorts something. Survey researchers reduce dozens of Likert-scale responses to a handful of factors. Whenever data lives in more dimensions than you can plot, some form of dimensionality reduction is the answer.

A few things worth noting:

- The new axes (principal components) are not the original features. They are *combinations* of all four measurements, chosen to maximize spread.
- We standardize before running PCA for the same reason we standardize before k-means: if one feature has a much larger scale, it will dominate the spread calculation and the first principal component will mostly just point in that feature's direction.
- PCA does not use the species labels. Like k-means, it is unsupervised — it only looks at the geometry of the data.

**Question.** In Lecture 3 we made scatter plots by picking two features to put on the axes. How is that different from what PCA does? Which approach do you think would give a better picture of the cluster structure, and why?

In [ ]:
# Illustration: projecting 3D points onto a 2D plane — the core idea behind PCA
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
n_pts = 40

# Generate a tilted cluster in 3D — most variance along one direction
t = np.linspace(0, 1, n_pts)
pts = np.column_stack([
    2*t + 0.3*np.random.randn(n_pts),
    1.5*t + 0.3*np.random.randn(n_pts),
    0.3*np.random.randn(n_pts)          # thin in z — variance lives in x-y plane
])

# Shadow: project onto z=0 plane (drop z coordinate)
shadow = pts.copy()
shadow[:, 2] = -1.2   # flatten to z = -1.2 for visual separation

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

# 3D points
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
           color='steelblue', s=40, alpha=0.9, label='Data (3D)')

# Drop lines from each point to its shadow
for p, s in zip(pts, shadow):
    ax.plot([p[0], s[0]], [p[1], s[1]], [p[2], s[2]],
            color='gray', lw=0.5, alpha=0.5)

# Shadow points on the z=-1.2 plane
ax.scatter(shadow[:, 0], shadow[:, 1], shadow[:, 2],
           color='tomato', s=30, alpha=0.6, label='Shadow (2D projection)')

# Shade the projection plane
xx, yy = np.meshgrid(np.linspace(-0.5, 2.5, 2), np.linspace(-0.5, 2.0, 2))
ax.plot_surface(xx, yy, -1.2 * np.ones_like(xx),
                alpha=0.12, color='tomato')

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
ax.set_title('3D data projected onto a 2D plane\n(the shadow is the PCA view)')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

## 5.5 PCA with Scikit-learn

Finding the principal components requires solving a variance-maximization problem that involves the **covariance matrix** of $X_\text{std}$ — a $4 \times 4$ matrix that encodes how the four features vary together. Computing its eigenvectors gives the principal component directions. We will not derive this in full here (it belongs with the deeper linear algebra of Lecture 9), but `sklearn`'s `PCA` does it for us exactly.

What `PCA(n_components=2).fit_transform(X_std)` returns is the $150 \times 2$ matrix of projection coordinates. Each row $\mathbf{v}_i \in \mathbb{R}^2$ is flower $i$ projected into the plane of maximum variance — a 2D representation we can actually plot.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
V = pca.fit_transform(X_std)   # V has shape (150, 2)

print('V shape:', V.shape)
print(f'First 3 rows of V:')
print(np.round(V[:3, :], 4))

In [ ]:
# How much variance does each component explain?
evr = pca.explained_variance_ratio_
print(f'PC1 explains {evr[0]*100:.1f}% of total variance')
print(f'PC2 explains {evr[1]*100:.1f}% of total variance')
print(f'Together:    {sum(evr)*100:.1f}%')

**Question.** Two dimensions explain a large fraction of the total variance. What does this tell you about the intrinsic dimensionality of the Iris dataset — even though the data matrix $X$ has 4 columns?

## 5.6 Visualizing k-Means Results via PCA

Now we have everything we need: k-means cluster labels from Section 5.2, and PCA coordinates from Section 5.5. Plotting $Y$ colored by cluster label gives us a 2D picture of the 4D cluster structure.

We'll place two plots side by side:

1. PCA coordinates colored by **k-means cluster label** — what the algorithm found without labels
2. PCA coordinates colored by **true species label** — the ground truth

In [ ]:
# Align k-means cluster colors to true species colors by majority vote
# For each k-means cluster, find the species that contributes the most points
color_map = {}
for c in range(3):
    counts = [int(np.sum((labels == c) & (y == k))) for k in range(3)]
    color_map[c] = int(np.argmax(counts))   # species code this cluster best matches

# Build code -> name lookup (codes assigned alphabetically by pd.Categorical)
cat = pd.Categorical(df['species'])
species_by_code = {i: name for i, name in enumerate(cat.categories)}

colors = ['steelblue', 'tomato', 'seagreen']   # indexed by species code 0/1/2
cluster_colors = [colors[color_map[c]] for c in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: k-means clusters — color matched to best-matching species
ax = axes[0]
for c in range(3):
    mask = (labels == c)
    matched_name = species_by_code[color_map[c]]
    ax.scatter(V[mask, 0], V[mask, 1], color=cluster_colors[c],
               label=f'Cluster {c} ({matched_name})', alpha=0.7,
               edgecolors='white', linewidth=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('k-Means clusters (PCA view)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# Plot 2: true species — same color per species as Plot 1
ax = axes[1]
for k in range(3):
    mask = (y == k)
    ax.scatter(V[mask, 0], V[mask, 1], color=colors[k],
               label=species_by_code[k], alpha=0.7,
               edgecolors='white', linewidth=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('True species (PCA view)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Iris: k-Means Clusters vs. True Species (PCA view)', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** The cluster assignments (left) and the true species (right) should look very similar — the color alignment makes mismatches easy to spot. Where do they differ?

**Question.** Compare the two plots above. The left plot shows what k-means found without any labels; the right shows the true species. Where do the cluster assignments and the true labels differ? Cross-reference with the cross-tabulation from Section 5.2.

## 5.7 Why Standardize Before k-Means?

k-means assigns each point to its nearest centroid using Euclidean distance — $\|\mathbf{x}_i - \mathbf{c}_j\|$. This means that features with **larger numerical ranges will dominate the distance calculation**, regardless of whether they are actually more informative. 

**Question.** While larger numerical ranges have a greater affect on the distance calculation, what will be more influential in *assignment to groups*: the "smaller" scaled variable or the larger one?

Recall from Lecture 3: before standardization, the four Iris features had very different norms. A 1 cm difference in sepal length and a 1 cm difference in petal width contribute identically to the raw distance — but those features have very different natural scales of variation.

In [ ]:
# Run k-means on the RAW (unstandardized) data and compare
kmeans_raw = KMeans(n_clusters=3, n_init=10, random_state=0)
labels_raw = kmeans_raw.fit_predict(X)

print('Raw data clustering:')
for k, name in enumerate(species_names):
    row = [int(np.sum((y == k) & (labels_raw == j))) for j in range(3)]
    print(f'{name:>15}: {row}')

print()
print('Standardized data clustering:')
for k, name in enumerate(species_names):
    row = [int(np.sum((y == k) & (labels == j))) for j in range(3)]
    print(f'{name:>15}: {row}')

**Question.** Do the two clusterings agree? If the raw clustering does worse, which features do you think dominated the distance calculation — the ones with larger or smaller standard deviations?

**Looking ahead.** Lectures 4 and 5 introduced two fundamental unsupervised techniques: k-means and PCA. In Homework 4 you will apply both to the Palmer Penguins dataset — standardize, cluster, reduce, visualize, and evaluate. In later lectures we'll return to PCA more formally when we have the tools (eigenvalues and SVD) to derive why the principal component directions are what they are.